In [ ]:
from pathlib import Path

import pandas as pd

from application.main import classify_CAS_sections
from application.pipeline import classify_CAS_claude_single_file
from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CAS_CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "CAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.4")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_CAS_classification_registry = pd.read_csv(CAS_CLASSIFICATION_REGISTRY)

In [ ]:
df_CAS_classification_registry.duplicated().sum()

In [ ]:
# paths referenced in the code below use absolute paths relative to the original local environment


dependency_set = set(df_base_registry.loc[df_base_registry["output_sha"].isin(
    df_CAS_classification_registry.loc[df_CAS_classification_registry["method"]=="LLM", "output_sha"]),
    "dependencies"
])

cleaned_CAS_path_1 = Path("data/records/software_version/v1.0.4/pipeline_version/v1.0.0/10.1007_JHEP02(2024)136/CAS_section_cleaned.txt"
)

cleaned_CAS_path_2 = Path("data/records/software_version/v1.0.4/pipeline_version/v1.0.0/10.1007_JHEP04(2024)038/CAS_section_cleaned.txt"
)

cleaned_CAS_path_3 = Path("data/records/software_version/v1.0.4/pipeline_version/v1.0.0/10.1007_JHEP06(2024)163/CAS_section_cleaned.txt"
)

doc_doi_1 = "10.1007/JHEP02(2024)136"

doc_doi_2 = "10.1007/JHEP04(2024)038"

doc_doi_3 = "10.1007/JHEP06(2024)163"

df_base_registry, df_CAS_classification_registry, dependency_sha_1 = classify_CAS_claude_single_file(cleaned_CAS_path_1, 
                                                                                                   doc_doi_1,
                                                                                                   df_base_registry,
                                                                                                   df_CAS_classification_registry,
                                                                                                   version_object,
                                                                                                   dependency_set)

dependency_set.add(dependency_sha_1)

df_base_registry, df_CAS_classification_registry, dependency_sha_2 = classify_CAS_claude_single_file(cleaned_CAS_path_2, 
                                                                                                   doc_doi_2,
                                                                                                   df_base_registry,
                                                                                                   df_CAS_classification_registry,
                                                                                                   version_object,
                                                                                                   dependency_set)

dependency_set.add(dependency_sha_2)

df_base_registry, df_CAS_classification_registry, dependency_sha_3 = classify_CAS_claude_single_file(cleaned_CAS_path_3, 
                                                                                                   doc_doi_3,
                                                                                                   df_base_registry,
                                                                                                   df_CAS_classification_registry,
                                                                                                   version_object,
                                                                                                   dependency_set)

dependency_set.add(dependency_sha_3)

df_base_registry, df_CAS_classification_registry, dependency_sha_3 = classify_CAS_claude_single_file(cleaned_CAS_path_3, 
                                                                                                   doc_doi_3,
                                                                                                   df_base_registry,
                                                                                                   df_CAS_classification_registry,
                                                                                                   version_object,
                                                                                                   dependency_set)

display(df_CAS_classification_registry)

In [ ]:
classify_CAS_sections(df_base_registry, df_extraction_registry, df_CAS_classification_registry, version_object)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

counts = df_CAS_classification_registry.value_counts(subset="MCA_label")
total = counts.sum()
percentages = (counts/total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#888780', '#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig = plt.figure(figsize=(14,5))
fig.suptitle(f"CAS classification results - Manuscript Code Availability (n={total:,})", fontsize=13, fontweight='500', x=0.02, ha='left')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

card = [
    ("total sections", f"{total:,}", "#333"),
    ("no_code", f"{percentages.get('no_code', 0):.1f}%", '#888780'),
    ("open access", f"{percentages.get('open_access', 0):.1f}%", "#1D9E75"),
    ("restricted accesss", f"{percentages.get('restricted_access', 0):.1f}%", "#D85A30"),
]

ax_cards = fig.add_subplot(gs[0,0])

ax_cards.axis('off')
for i, (label, val, color) in enumerate(card):
    y = 0.85 - i * 0.22
    ax_cards.text(0.05, y,      label, fontsize=9,  color='gray',  transform=ax_cards.transAxes)
    ax_cards.text(0.05, y-0.09, val,   fontsize=16, color=color, fontweight='bold', transform=ax_cards.transAxes)

ax_donut = fig.add_subplot(gs[0, 1])
pos = ax_donut.get_position()
ax_donut.set_position([pos.x0 - 0.1, pos.y0, pos.width, pos.height])
wedges, _ = ax_donut.pie(values, colors=colors, startangle=90,
                         wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5))
ax_donut.set_title("Manuscript Code Availability category distribution")

ax_bar = fig.add_subplot(gs[0, 2])
bars = ax_bar.barh(labels[::-1], percentages.values[::-1], color=colors[::-1], height=0.6)
ax_bar.set_xlabel("% of sections", fontsize=9, color='gray')
ax_bar.tick_params(axis='y', labelsize=9)
ax_bar.tick_params(axis='x', labelsize=9, colors='gray')
ax_bar.spines[["top", "right", "left"]].set_visible(False)
ax_bar.xaxis.grid(True, linestyle='--', alpha=0.4)
ax_bar.set_axisbelow(True)

for bar, pct, count in zip(bars, percentages.values[::-1], values[::-1]):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{pct}%, ({count:,})", va='center', fontsize=8, color='gray')
    
ax_bar.set_xlim(0, percentages.max() + 15)

plt.tight_layout()
plt.savefig("MCA_classification_v104.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

has_code = df_CAS_classification_registry[df_CAS_classification_registry["MCA_label"] != "no_code"]
counts = has_code.value_counts(subset="MCA_label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle(
    f"MCA classification — studies with associated author-produced code  (n={total:,} / {len(df_CAS_classification_registry):,} total)",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.55)

ax.set_xlabel("% of sections with code", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, pcts.max() + 18)

plt.tight_layout()
plt.savefig("MCA_classification_with_code.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

counts = df_CAS_classification_registry.value_counts(subset="ETA_label")
total = counts.sum()
percentages = (counts/total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#888780', '#D85A30', '#1D9E75', '#7F77DD', '#FAC775']

fig = plt.figure(figsize=(14,5))
fig.suptitle(f"CAS classification results - External Tool Availability (n={total:,})", fontsize=13, fontweight='500', x=0.02, ha='left')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

card = [
    ("total sections", f"{total:,}", "#333"),
    ("no tool", f"{percentages.get('no_tool', 0):.1f}%", '#888780'),
    ("open tool", f"{percentages.get('open_tool', 0):.1f}%", "#1D9E75"),
    ("proprietary tool", f"{percentages.get('proprietary_tool', 0):.1f}%", "#D85A30"),
]

ax_cards = fig.add_subplot(gs[0,0])

ax_cards.axis('off')
for i, (label, val, color) in enumerate(card):
    y = 0.85 - i * 0.22
    ax_cards.text(0.05, y,      label, fontsize=9,  color='gray',  transform=ax_cards.transAxes)
    ax_cards.text(0.05, y-0.09, val,   fontsize=16, color=color, fontweight='bold', transform=ax_cards.transAxes)

ax_donut = fig.add_subplot(gs[0, 1])
pos = ax_donut.get_position()
ax_donut.set_position([pos.x0 - 0.1, pos.y0, pos.width, pos.height])
wedges, _ = ax_donut.pie(values, colors=colors, startangle=90,
                         wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5))
ax_donut.set_title("External Tool Availability category distribution")

ax_bar = fig.add_subplot(gs[0, 2])
bars = ax_bar.barh(labels[::-1], percentages.values[::-1], color=colors[::-1], height=0.6)
ax_bar.set_xlabel("% of sections", fontsize=9, color='gray')
ax_bar.tick_params(axis='y', labelsize=9)
ax_bar.tick_params(axis='x', labelsize=9, colors='gray')
ax_bar.spines[["top", "right", "left"]].set_visible(False)
ax_bar.xaxis.grid(True, linestyle='--', alpha=0.4)
ax_bar.set_axisbelow(True)

for bar, pct, count in zip(bars, percentages.values[::-1], values[::-1]):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{pct}%, ({count:,})", va='center', fontsize=8, color='gray')
    
ax_bar.set_xlim(0, percentages.max() + 15)

plt.tight_layout()
plt.savefig("ETA_classification_v104.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

has_code = df_CAS_classification_registry[df_CAS_classification_registry["ETA_label"] != "no_tool"]
counts = has_code.value_counts(subset="ETA_label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle(
    f"ETA classification — studies with associated external software tools  (n={total:,} / {len(df_CAS_classification_registry):,} total)",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.55)

ax.set_xlabel("% of sections with associated EST", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, pcts.max() + 18)

plt.tight_layout()
plt.savefig("ETA_classification_with_tools.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

df_cas_journal = (
    df_CAS_classification_registry
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "journal"]], on="doc_doi", how="left")
)
df_cas_journal["journal"] = df_cas_journal["journal"].fillna("Unknown journal")

journal_order = df_cas_journal["journal"].value_counts().index.tolist()
mca_order = [
    "no_code",
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
mca_colors = {
    "no_code": "#888780",
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

mca_by_journal = (
    pd.crosstab(df_cas_journal["journal"], df_cas_journal["MCA_label"], normalize="index")
    .reindex(index=journal_order)
    .reindex(columns=mca_order, fill_value=0)
    * 100
).round(1)
journal_totals = df_cas_journal["journal"].value_counts().reindex(journal_order)

fig, ax = plt.subplots(figsize=(10, 4.8))
left = pd.Series(0.0, index=mca_by_journal.index)

for label in mca_order:
    widths = mca_by_journal[label]
    ax.barh(
        mca_by_journal.index,
        widths,
        left=left,
        color=mca_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of CAS sections", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for journal, total in journal_totals.items():
    ax.text(101.5, journal, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 110)
ax.legend(title="MCA label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle("CAS classification by journal - Manuscript Code Availability", fontsize=12, x=0.02, ha="left")

plt.tight_layout()
plt.savefig("MCA_classification_by_journal.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

df_cas_journal = (
    df_CAS_classification_registry
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "journal"]], on="doc_doi", how="left")
)
df_cas_journal["journal"] = df_cas_journal["journal"].fillna("Unknown journal")

journal_order = df_cas_journal["journal"].value_counts().index.tolist()
eta_order = [
    "no_tool",
    "proprietary_tool",
    "open_tool",
    "mixed_tool",
    "unknown_status_tool",
]
eta_colors = {
    "no_tool": "#888780",
    "proprietary_tool": "#D85A30",
    "open_tool": "#1D9E75",
    "mixed_tool": "#7F77DD",
    "unknown_status_tool": "#FAC775",
}

eta_by_journal = (
    pd.crosstab(df_cas_journal["journal"], df_cas_journal["ETA_label"], normalize="index")
    .reindex(index=journal_order)
    .reindex(columns=eta_order, fill_value=0)
    * 100
).round(1)
journal_totals = df_cas_journal["journal"].value_counts().reindex(journal_order)

fig, ax = plt.subplots(figsize=(10, 4.8))
left = pd.Series(0.0, index=eta_by_journal.index)

for label in eta_order:
    widths = eta_by_journal[label]
    ax.barh(
        eta_by_journal.index,
        widths,
        left=left,
        color=eta_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of CAS sections", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for journal, total in journal_totals.items():
    ax.text(101.5, journal, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 110)
ax.legend(title="ETA label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle("CAS classification by journal - External Tool Availability", fontsize=12, x=0.02, ha="left")

plt.tight_layout()
plt.savefig("ETA_classification_by_journal.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

combo_counts = (
    df_CAS_classification_registry
    .assign(combo_label=lambda df: df["MCA_label"] + " | " + df["ETA_label"])
    ["combo_label"]
    .value_counts()
)
combo_pcts = (combo_counts / combo_counts.sum() * 100).round(1)

fig_height = max(4.5, 0.38 * len(combo_counts) + 1.5)
fig, ax = plt.subplots(figsize=(11, fig_height))
bars = ax.barh(combo_counts.index[::-1], combo_pcts.values[::-1], color="#5B7DB1", height=0.68)

ax.set_xlabel("% of CAS sections", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for bar, pct, count in zip(bars, combo_pcts.values[::-1], combo_counts.values[::-1]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{pct}%  ({count:,})",
        va="center",
        fontsize=9,
        color="gray",
    )

ax.set_xlim(0, combo_pcts.max() + 14)
fig.suptitle(
    f"CAS classification - combined MCA/ETA label distribution (n={combo_counts.sum():,})",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("CAS_combined_MCA_ETA_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

has_est = df_CAS_classification_registry[df_CAS_classification_registry["ETA_label"] != "no_tool"]
mca_est_order = [
    "no_code",
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
mca_est_colors = {
    "no_code": "#888780",
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

mca_est_counts = has_est["MCA_label"].value_counts().reindex(mca_est_order, fill_value=0)
mca_est_pcts = (mca_est_counts / mca_est_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(
    mca_est_counts.index[::-1],
    mca_est_pcts.values[::-1],
    color=[mca_est_colors[label] for label in mca_est_counts.index[::-1]],
    height=0.62,
)

ax.set_xlabel("% of CAS sections with mentioned EST", fontsize=9, color="gray")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for bar, pct, count in zip(bars, mca_est_pcts.values[::-1], mca_est_counts.values[::-1]):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{pct}%  ({count:,})",
        va="center",
        fontsize=9,
        color="gray",
    )

ax.set_xlim(0, mca_est_pcts.max() + 16)
fig.suptitle(
    f"MCA distribution among CAS sections with mentioned EST (n={mca_est_counts.sum():,} / {len(df_CAS_classification_registry):,} total)",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("CAS_MCA_distribution_with_EST.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mca_scatter_order = [
    "no_code",
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
eta_scatter_order = [
    "no_tool",
    "open_tool",
    "proprietary_tool",
    "mixed_tool",
    "unknown_status_tool",
]
mca_scatter_colors = {
    "no_code": "#888780",
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

scatter_df = df_CAS_classification_registry[
    df_CAS_classification_registry["MCA_label"].isin(mca_scatter_order)
    & df_CAS_classification_registry["ETA_label"].isin(eta_scatter_order)
].copy()

mca_map = {label: idx for idx, label in enumerate(mca_scatter_order)}
eta_map = {label: idx for idx, label in enumerate(eta_scatter_order)}
rng = np.random.default_rng(42)
jitter_width = 0.18

scatter_df["x"] = scatter_df["MCA_label"].map(mca_map) + rng.uniform(-jitter_width, jitter_width, len(scatter_df))
scatter_df["y"] = scatter_df["ETA_label"].map(eta_map) + rng.uniform(-jitter_width, jitter_width, len(scatter_df))
scatter_df["color"] = scatter_df["MCA_label"].map(mca_scatter_colors)

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.scatter(
    scatter_df["x"],
    scatter_df["y"],
    c=scatter_df["color"],
    s=22,
    alpha=0.28,
    linewidths=0,
)

combo_counts = (
    scatter_df.groupby(["MCA_label", "ETA_label"])
    .size()
    .reset_index(name="count")
)
for row in combo_counts.itertuples(index=False):
    ax.text(
        mca_map[row.MCA_label],
        eta_map[row.ETA_label] + 0.29,
        f"n={row.count:,}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#3A3A3A",
    )

ax.set_xticks(range(len(mca_scatter_order)))
ax.set_xticklabels(mca_scatter_order, rotation=25, ha="right")
ax.set_yticks(range(len(eta_scatter_order)))
ax.set_yticklabels(eta_scatter_order)
ax.set_xlim(-0.5, len(mca_scatter_order) - 0.5)
ax.set_ylim(-0.5, len(eta_scatter_order) - 0.5)
ax.grid(True, linestyle="--", alpha=0.25)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlabel("MCA label", fontsize=10)
ax.set_ylabel("ETA label", fontsize=10)
fig.suptitle(
    f"CAS classification - MCA vs ETA label clouds (jittered scatter, n={len(scatter_df):,})",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("CAS_MCA_vs_ETA_jitter_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

df_cas_journal = (
    df_CAS_classification_registry
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "journal"]], on="doc_doi", how="left")
)
df_cas_journal["journal"] = df_cas_journal["journal"].fillna("Unknown journal")

mca_order_no_code = [
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
mca_colors_no_code = {
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

df_cas_with_code = df_cas_journal[df_cas_journal["MCA_label"] != "no_code"].copy()
journal_order_no_code = df_cas_with_code["journal"].value_counts().index.tolist()

mca_by_journal_no_code = (
    pd.crosstab(df_cas_with_code["journal"], df_cas_with_code["MCA_label"], normalize="index")
    .reindex(index=journal_order_no_code)
    .reindex(columns=mca_order_no_code, fill_value=0)
    * 100
).round(1)
journal_totals_no_code = df_cas_with_code["journal"].value_counts().reindex(journal_order_no_code)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
left = pd.Series(0.0, index=mca_by_journal_no_code.index)

for label in mca_order_no_code:
    widths = mca_by_journal_no_code[label]
    ax.barh(
        mca_by_journal_no_code.index,
        widths,
        left=left,
        color=mca_colors_no_code[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of MCA-labeled sections, excluding no_code", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for journal, total in journal_totals_no_code.items():
    ax.text(101.5, journal, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 112)
ax.legend(title="MCA label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle(
    f"MCA category proportions by journal, excluding no_code (n={len(df_cas_with_code):,})",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("MCA_classification_by_journal_excluding_no_code.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

if "df_cas_journal" not in globals():
    df_cas_journal = (
        df_CAS_classification_registry
        .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
        .merge(df_document_registry[["doc_doi", "journal"]], on="doc_doi", how="left")
    )
df_cas_journal["journal"] = df_cas_journal["journal"].fillna("Unknown journal")

if "mca_order_no_code" not in globals():
    mca_order_no_code = [
        "restricted_access",
        "open_access",
        "unclear",
        "partial_access",
        "no_access",
        "incorrect",
    ]

df_cas_with_code = df_cas_journal[df_cas_journal["MCA_label"] != "no_code"].copy()
if "journal_order_no_code" not in globals():
    journal_order_no_code = df_cas_with_code["journal"].value_counts().index.tolist()

mca_counts_by_journal = (
    pd.crosstab(df_cas_with_code["journal"], df_cas_with_code["MCA_label"])
    .reindex(index=journal_order_no_code)
    .reindex(columns=mca_order_no_code, fill_value=0)
)
journal_totals_no_code = mca_counts_by_journal.sum(axis=1)

count_cmap = LinearSegmentedColormap.from_list(
    "mca_counts",
    ["#f7f4ea", "#d7d1bc", "#9c9485", "#5c554c"],
)

fig_width = max(10, 1.4 * len(mca_counts_by_journal.columns) + 3)
fig_height = max(4.8, 0.8 * len(mca_counts_by_journal.index) + 1.8)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

im = ax.imshow(mca_counts_by_journal.values, cmap=count_cmap, aspect="auto")

ax.set_xticks(range(len(mca_counts_by_journal.columns)))
ax.set_xticklabels(mca_counts_by_journal.columns, rotation=25, ha="right", fontsize=9)
ax.set_yticks(range(len(mca_counts_by_journal.index)))
ax.set_yticklabels(mca_counts_by_journal.index, fontsize=9)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

max_value = mca_counts_by_journal.values.max() if mca_counts_by_journal.values.size else 0
for i, journal in enumerate(mca_counts_by_journal.index):
    for j, label in enumerate(mca_counts_by_journal.columns):
        value = int(mca_counts_by_journal.iloc[i, j])
        text_color = "white" if max_value and value > max_value * 0.45 else "#2f2a24"
        ax.text(j, i, f"{value}", ha="center", va="center", color=text_color, fontsize=9, fontweight="bold")

for i, total in enumerate(journal_totals_no_code):
    ax.text(
        len(mca_counts_by_journal.columns) - 0.05,
        i,
        f" total={int(total):,}",
        ha="left",
        va="center",
        fontsize=9,
        color="gray",
        clip_on=False,
    )

ax.set_xticks([x - 0.5 for x in range(1, len(mca_counts_by_journal.columns))], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, len(mca_counts_by_journal.index))], minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.ax.tick_params(labelsize=8, colors="gray")
cbar.outline.set_visible(False)
cbar.set_label("count", fontsize=9, color="gray")

fig.suptitle("MCA category counts by journal, excluding no_code", fontsize=12, x=0.02, ha="left")
plt.tight_layout()
plt.savefig("MCA_classification_by_journal_counts_excluding_no_code.png", dpi=150, bbox_inches="tight")
plt.show()
